# Structure-Aware Chunking

Exploits the **inherent structural boundaries** of each source, discovered by analysing the actual corpus files.

| Source | Key finding | Strategy |
|--------|-------------|----------|
| **Wikipedia** (223 docs) | `metadata.sections` names appear as the **first line of each `\n\n`-separated paragraph** (74.5 % match rate). `==heading==` markers are stripped by the API. | Match paragraph first-lines against `metadata.sections` set to detect section boundaries. |
| **Wikibooks** (78 docs) | Fixed recipe keywords (`Ingredients`, `Procedure`, `Notes`…) appear as standalone lines in 97–100 % of docs. | Split on these keyword lines; each recipe field → one chunk. |
| **Blog** (30 docs) | Each doc is already one `<h3>` dish section (avg ~400 tokens). | Keep whole; split by paragraph only when over `MAX_TOKENS`. |

**Three improvements over a naïve approach (validated on the real data):**
1. **`Gallery` added to skip-list** — in this corpus every Gallery paragraph contains only `See also` links and boilerplate, never real content.
2. **`List of …` articles kept whole** — these Wikipedia articles use single-letter alphabetic headings (`B`, `C`, `D`…) as section names; splitting there produces meaningless chunks.
3. **Boilerplate body filter** — even after section-name matching, some bodies consist entirely of `See also / References / List of …` lines (residuals from the API export); these are dropped by a content check.

**Two runs:**
1. **Coarse** — section-level split, no post-processing → `chunks_structure_aware_coarse.jsonl`
2. **Fine** — additionally merges sections < `MIN_TOKENS` (40) into the next sibling → `chunks_structure_aware_fine.jsonl`

In [1]:
%pip install tiktoken

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import os
import re
import tiktoken
from typing import List, Dict, Any, Tuple
from collections import Counter

In [3]:
# Config
CORPUS_PATHS = [
    "../data/corpus_wikipedia.jsonl",
    "../data/corpus_wikibooks_east_asia.jsonl",
    "../data/corpus_blog.jsonl",
]

MAX_TOKENS = 400   # sections exceeding this are split further by paragraph
MIN_TOKENS = 40    # fine mode: sections below this are merged into the next

# Wikipedia: section headings whose content is always boilerplate
SKIP_SECTIONS = {
    "See also", "References", "External links", "Further reading",
    "Notes", "Bibliography", "Footnotes",
    "Gallery",   # empirically: always contains only See-also links in this corpus
}

# Wikipedia: single uppercase letter = alphabetic-index heading (List-of articles)
ALPHABET_RE = re.compile(r"^[A-Z]$")

# Wikipedia: body text starting with these strings is residual boilerplate
BOILERPLATE_STARTS = (
    "See also", "References", "External links", "List of", "Media related",
)

# Wikibooks: fixed recipe section keywords (appear as standalone lines)
WIKIBOOKS_SECTION_KEYWORDS = [
    "Ingredients", "Procedure", "Notes, tips, and variations",
    "Notes", "Warnings", "Equipment", "Variations",
]

ENCODING = tiktoken.get_encoding("cl100k_base")

## Load Corpus

In [4]:
def load_corpus(paths: List[str]) -> List[Dict[str, Any]]:
    documents = []
    for path in paths:
        if not os.path.exists(path):
            print(f"[WARN] File not found, skipped: {path}")
            continue
        count = 0
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    documents.append(json.loads(line))
                    count += 1
        print(f"Loaded {count} docs from {path}")
    print(f"Total: {len(documents)} documents")
    return documents

documents = load_corpus(CORPUS_PATHS)

Loaded 223 docs from ../data/corpus_wikipedia.jsonl
Loaded 78 docs from ../data/corpus_wikibooks_east_asia.jsonl
Loaded 30 docs from ../data/corpus_blog.jsonl
Total: 331 documents


## Chunking Functions

In [5]:
def count_tokens(text: str) -> int:
    return len(ENCODING.encode(text))


def detect_source_type(doc: Dict[str, Any]) -> str:
    """Returns 'blog', 'wikibooks', or 'wikipedia'."""
    doc_id = doc.get("doc_id", "")
    if doc_id.startswith("blog_"):
        return "blog"
    if "wikibooks" in doc.get("source", "").lower():
        return "wikibooks"
    return "wikipedia"


def is_boilerplate_body(body: str) -> bool:
    """True if body is entirely residual boilerplate (See also, References…)."""
    return any(body.startswith(prefix) for prefix in BOILERPLATE_STARTS)


def split_by_paragraph(text: str) -> List[str]:
    """
    Fallback for oversized sections: greedily accumulate paragraphs
    (split on double-newline) until MAX_TOKENS is reached, then start
    a new chunk. A single paragraph that still exceeds MAX_TOKENS is
    trimmed at the token boundary.
    """
    paragraphs = [p.strip() for p in re.split(r"\n{2,}", text) if p.strip()]
    chunks, buffer = [], ""
    for para in paragraphs:
        candidate = (buffer + "\n\n" + para).strip() if buffer else para
        if count_tokens(candidate) <= MAX_TOKENS:
            buffer = candidate
        else:
            if buffer:
                chunks.append(buffer)
            if count_tokens(para) > MAX_TOKENS:
                tokens  = ENCODING.encode(para)
                trimmed = ENCODING.decode(tokens[:MAX_TOKENS]).strip()
                chunks.append(trimmed)
                buffer = ""
            else:
                buffer = para
    if buffer:
        chunks.append(buffer)
    return chunks if chunks else [text.strip()]

In [6]:
def parse_wikipedia_sections(
    text: str,
    section_names: List[str],
    doc_title: str = "",
) -> List[Tuple[str, str]]:
    """
    Parse Wikipedia plain-text into (section_heading, body) pairs.

    Background
    ----------
    The `wikipediaapi` library strips `==heading==` wikitext markers,
    but section names are preserved as the FIRST LINE of each
    double-newline-separated paragraph (confirmed: 74.5 % match rate
    across all paragraphs in this corpus).

    Algorithm
    ---------
    1.  'List of …' articles use single-letter alphabetic headings
        (B, C, D …); splitting there is meaningless → return whole text.
    2.  Build the valid section set: metadata.sections minus SKIP_SECTIONS
        minus single-uppercase-letter headings.
    3.  Scan paragraphs; when the first line matches the section set,
        flush the current buffer and start a new section.
    4.  Drop bodies that are purely boilerplate (See also, References…).

    Returns list of (heading, body) tuples.
    Lead section (before any named heading) has heading = "".
    """
    # Special case: List-of articles → keep whole
    if doc_title.startswith("List of"):
        return [("", text.strip())]

    sections_set = {
        s for s in set(section_names) - SKIP_SECTIONS
        if not ALPHABET_RE.match(s)
    }

    paragraphs = [p.strip() for p in re.split(r"\n{2,}", text) if p.strip()]
    result: List[Tuple[str, str]] = []
    current_heading = ""
    current_body_parts: List[str] = []

    for para in paragraphs:
        first_line = para.split("\n")[0].strip()

        if first_line in sections_set:
            # Flush previous section
            body = "\n\n".join(current_body_parts).strip()
            if body and not is_boilerplate_body(body):
                result.append((current_heading, body))
            # Start new section
            current_heading    = first_line
            rest               = para[len(first_line):].strip()
            current_body_parts = [rest] if rest else []
        else:
            current_body_parts.append(para)

    # Flush last section
    body = "\n\n".join(current_body_parts).strip()
    if body and not is_boilerplate_body(body):
        result.append((current_heading, body))

    return result if result else [("", text.strip())]

In [7]:
def parse_wikibooks_sections(text: str) -> List[Tuple[str, str]]:
    """
    Parse a Wikibooks recipe into (section_heading, body) pairs.

    Wikibooks recipes in this corpus contain fixed keyword headings
    (Ingredients, Procedure, Notes…) as standalone lines in 97–100 %
    of documents. Each keyword field becomes one chunk.

    Returns list of (heading, body) tuples.
    The recipe intro paragraph has heading = "Intro".
    """
    kw_pattern = re.compile(
        r"^(?P<kw>" + "|".join(re.escape(k) for k in WIKIBOOKS_SECTION_KEYWORDS) + r")\s*$",
        re.MULTILINE,
    )
    matches = list(kw_pattern.finditer(text))

    if not matches:
        return [("Intro", text.strip())]

    result: List[Tuple[str, str]] = []

    intro = text[: matches[0].start()].strip()
    if intro:
        result.append(("Intro", intro))

    for i, m in enumerate(matches):
        heading    = m.group("kw")
        body_start = m.end()
        body_end   = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        body       = text[body_start:body_end].strip()
        if body:
            result.append((heading, body))

    return result if result else [("Intro", text.strip())]

In [8]:
def structure_aware_chunk(
    doc: Dict[str, Any],
    merge_short: bool = False,
) -> List[Dict[str, Any]]:
    """
    Chunk one document using source-specific structural boundaries.

    Parameters
    ----------
    doc          : one JSONL record
    merge_short  : if True (fine mode), prepend sections whose body is
                   < MIN_TOKENS to the following section before chunking

    Returns
    -------
    List of chunk dicts with the same schema as the fixed/semantic notebooks.
    """
    doc_id   = doc.get("doc_id", "unknown")
    text     = doc.get("text", "").strip()
    title    = doc.get("title", "")
    source   = doc.get("source", "")
    src_type = detect_source_type(doc)

    if not text:
        return []

    # ── Parse into (heading, body) pairs ──────────────────────────
    if src_type == "wikipedia":
        section_names = doc.get("metadata", {}).get("sections", [])
        sections = parse_wikipedia_sections(text, section_names, title)

    elif src_type == "wikibooks":
        sections = parse_wikibooks_sections(text)

    else:  # blog — already section-level
        section_title = doc.get("metadata", {}).get("section_title", "")
        sections = [(section_title, text)]

    # ── Fine mode: merge short sections into next sibling ─────────
    if merge_short and len(sections) > 1:
        merged: List[Tuple[str, str]] = []
        i = 0
        while i < len(sections):
            heading, body = sections[i]
            if count_tokens(body) < MIN_TOKENS and i + 1 < len(sections):
                next_h, next_b = sections[i + 1]
                sections[i + 1] = (next_h, body + "\n\n" + next_b)
            else:
                merged.append((heading, body))
            i += 1
        sections = merged

    # ── Build chunk records ───────────────────────────────────────
    chunks: List[Dict[str, Any]] = []
    chunk_idx = 0

    for heading, body in sections:
        chunk_title = f"{title} > {heading}".strip(" >") if heading else title

        text_blocks = (
            [body] if count_tokens(body) <= MAX_TOKENS
            else split_by_paragraph(body)
        )

        for block in text_blocks:
            if not block.strip():
                continue
            chunks.append({
                "chunk_id": f"{doc_id}_chunk_{chunk_idx}",
                "text":     block.strip(),
                "embedding": [],
                "metadata": {
                    "doc_id":          doc_id,
                    "source":          source,
                    "title":           chunk_title,
                    "section_heading": heading,
                },
            })
            chunk_idx += 1

    return chunks

In [9]:
def chunk_corpus(
    documents: List[Dict[str, Any]],
    merge_short: bool = False,
) -> List[Dict[str, Any]]:
    """Chunk all documents and build output records."""
    all_chunks = []
    for doc in documents:
        all_chunks.extend(structure_aware_chunk(doc, merge_short=merge_short))
    return all_chunks


def save_chunks(chunks: List[Dict], path: str):
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for c in chunks:
            f.write(json.dumps(c, ensure_ascii=False) + "\n")
    size_mb = os.path.getsize(path) / (1024 * 1024)
    print(f"Saved {len(chunks)} chunks to {path} ({size_mb:.2f} MB)")


def print_stats(chunks: List[Dict], label: str):
    token_counts  = [count_tokens(c["text"]) for c in chunks]
    source_counts = Counter(c["metadata"]["source"] for c in chunks)
    print(f"\n--- {label} ---")
    print(f"Total chunks : {len(chunks)}")
    print(f"Tokens  avg={sum(token_counts)/max(len(token_counts),1):.0f}  "
          f"min={min(token_counts, default=0)}  max={max(token_counts, default=0)}")
    for src, cnt in source_counts.most_common():
        print(f"  {src}: {cnt}")

## Run 1: Coarse (section-level, no merging)

In [10]:
chunks_coarse = chunk_corpus(documents, merge_short=False)
print_stats(chunks_coarse, "Structure-aware, coarse (no merge)")
save_chunks(chunks_coarse, "data/chunks_structure_aware_coarse.jsonl")


--- Structure-aware, coarse (no merge) ---
Total chunks : 2607
Tokens  avg=187  min=1  max=400
  Wikipedia: 2297
  Wikibooks: 280
  Around the World in 80 Cuisines Blog: 30
Saved 2607 chunks to data/chunks_structure_aware_coarse.jsonl (2.49 MB)


## Run 2: Fine (short sections merged into next)

In [11]:
chunks_fine = chunk_corpus(documents, merge_short=True)
print_stats(chunks_fine, "Structure-aware, fine (merge short < 40 tokens)")
save_chunks(chunks_fine, "data/chunks_structure_aware_fine.jsonl")


--- Structure-aware, fine (merge short < 40 tokens) ---
Total chunks : 2444
Tokens  avg=199  min=7  max=400
  Wikipedia: 2178
  Wikibooks: 236
  Around the World in 80 Cuisines Blog: 30
Saved 2444 chunks to data/chunks_structure_aware_fine.jsonl (2.46 MB)


In [12]:
t_c = [count_tokens(c["text"]) for c in chunks_coarse]
t_f = [count_tokens(c["text"]) for c in chunks_fine]

print(f"{'':35s} {'Coarse':>10s} {'Fine':>10s}")
print(f"{'Total chunks':35s} {len(chunks_coarse):>10d} {len(chunks_fine):>10d}")
print(f"{'Avg tokens/chunk':35s} {sum(t_c)/max(len(t_c),1):>10.0f} {sum(t_f)/max(len(t_f),1):>10.0f}")
print(f"{'Min tokens/chunk':35s} {min(t_c, default=0):>10d} {min(t_f, default=0):>10d}")
print(f"{'Max tokens/chunk':35s} {max(t_c, default=0):>10d} {max(t_f, default=0):>10d}")
print(f"{'Total tokens':35s} {sum(t_c):>10d} {sum(t_f):>10d}")

                                        Coarse       Fine
Total chunks                              2607       2444
Avg tokens/chunk                           187        199
Min tokens/chunk                             1          7
Max tokens/chunk                           400        400
Total tokens                            487029     487065


In [13]:
# Verify output format
for path in ["data/chunks_structure_aware_coarse.jsonl", "data/chunks_structure_aware_fine.jsonl"]:
    with open(path, "r", encoding="utf-8") as f:
        sample = json.loads(f.readline())
    print(f"\n[{os.path.basename(path)}]")
    print(json.dumps(sample, ensure_ascii=False, indent=2))


[chunks_structure_aware_coarse.jsonl]
{
  "chunk_id": "wiki_0001_chunk_0",
  "text": "East Asian cuisine has evolved with common usage of oils, fats and sauces in the preparation of dishes.",
  "embedding": [],
  "metadata": {
    "doc_id": "wiki_0001",
    "source": "Wikipedia",
    "title": "List of Asian cuisines",
    "section_heading": ""
  }
}

[chunks_structure_aware_fine.jsonl]
{
  "chunk_id": "wiki_0001_chunk_0",
  "text": "East Asian cuisine has evolved with common usage of oils, fats and sauces in the preparation of dishes.",
  "embedding": [],
  "metadata": {
    "doc_id": "wiki_0001",
    "source": "Wikipedia",
    "title": "List of Asian cuisines",
    "section_heading": ""
  }
}
